In [ ]:
# Run this cell first to install required packages in JupyterLite
%pip install -q numpy matplotlib ipywidgets scikit-learn


# Optimizer Comparison — Sections 5–8

Compare **Batch GD**, **Momentum**, **AdaGrad**, and **RMSProp** on an ill-conditioned loss landscape.

| Optimizer | Update rule | Key property |
|-----------|-------------|--------------|
| Batch GD  | $\theta \leftarrow \theta - \alpha \nabla L$ | Uniform steps; slow on narrow valleys |
| Momentum  | $v \leftarrow \beta v - \alpha \nabla L$; $\theta \leftarrow \theta + v$ | Accelerates through flat regions |
| AdaGrad   | $G \leftarrow G + (\nabla L)^2$; $\theta \leftarrow \theta - \frac{\alpha}{\sqrt{G+\varepsilon}}\nabla L$ | Per-param lr; decays to 0 |
| RMSProp   | $E \leftarrow \rho E + (1-\rho)(\nabla L)^2$; $\theta \leftarrow \theta - \frac{\alpha}{\sqrt{E+\varepsilon}}\nabla L$ | Like AdaGrad but lr stays alive |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets


# ── Loss surfaces ──────────────────────────────────────────────────────────────
def get_loss(surface, kappa):
    if surface == 'bowl':
        b = 1 / np.sqrt(kappa)
        return (
            lambda x, y: 0.5 * x**2 + 0.5 * (y / b)**2,
            lambda x, y: x,
            lambda x, y: y / b**2,
            (-1.8, -1.8), (-2.2, 2.2), (-2.2, 2.2),
        )
    elif surface == 'banana':
        return (
            lambda x, y: (1 - x)**2 + 100 * (y - x**2)**2,
            lambda x, y: -2 * (1 - x) - 400 * x * (y - x**2),
            lambda x, y: 200 * (y - x**2),
            (-1.2, 1.0), (-2.0, 2.0), (-1.0, 3.0),
        )
    else:  # saddle valley
        return (
            lambda x, y: x**2 / kappa - y**2 + 0.1 * y**4,
            lambda x, y: 2 * x / kappa,
            lambda x, y: -2 * y + 0.4 * y**3,
            (0.5, 0.2), (-3.0, 3.0), (-2.5, 2.5),
        )


# ── Optimizers ─────────────────────────────────────────────────────────────────
def run_optimizer(name, f, gx_fn, gy_fn, x0, y0, steps, lr, beta=0.9, rho=0.9, eps=1e-8):
    x, y = float(x0), float(y0)
    vx, vy = 0.0, 0.0
    Gx, Gy = 0.0, 0.0
    Ex, Ey = 1e-8, 1e-8
    path, losses = [(x, y)], [f(x, y)]

    for _ in range(steps):
        dgx, dgy = gx_fn(x, y), gy_fn(x, y)
        if name == 'BGD':
            dx, dy = -lr * dgx, -lr * dgy
        elif name == 'Momentum':
            vx = beta * vx - lr * dgx
            vy = beta * vy - lr * dgy
            dx, dy = vx, vy
        elif name == 'AdaGrad':
            Gx += dgx**2; Gy += dgy**2
            dx = -lr / np.sqrt(Gx + eps) * dgx
            dy = -lr / np.sqrt(Gy + eps) * dgy
        else:  # RMSProp
            Ex = rho * Ex + (1 - rho) * dgx**2
            Ey = rho * Ey + (1 - rho) * dgy**2
            dx = -lr / np.sqrt(Ex + eps) * dgx
            dy = -lr / np.sqrt(Ey + eps) * dgy
        x = np.clip(x + dx, -5, 5)
        y = np.clip(y + dy, -5, 5)
        path.append((x, y))
        losses.append(max(1e-10, f(x, y)))
    return np.array(path), np.array(losses)


def draw_comparison(surface='bowl', kappa=10, steps=200,
                    show_bgd=True, lr_bgd=0.03,
                    show_mom=True, lr_mom=0.03, beta=0.9,
                    show_ag=True, lr_ag=3.0,
                    show_rms=True, lr_rms=0.03, rho=0.9):

    f, gx_fn, gy_fn, (x0, y0), xlim, ylim = get_loss(surface, kappa)

    optimizers = []
    if show_bgd: optimizers.append(('BGD',      lr_bgd, {}))
    if show_mom: optimizers.append(('Momentum', lr_mom, {'beta': beta}))
    if show_ag:  optimizers.append(('AdaGrad',  lr_ag,  {}))
    if show_rms: optimizers.append(('RMSProp',  lr_rms, {'rho': rho}))

    colors = {'BGD': '#2563eb', 'Momentum': '#dc2626', 'AdaGrad': '#059669', 'RMSProp': '#d97706'}

    # Compute paths
    results = {}
    for name, lr, kw in optimizers:
        path, losses = run_optimizer(name, f, gx_fn, gy_fn, x0, y0, steps, lr, **kw)
        results[name] = (path, losses)

    # Plots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Contour
    xs = np.linspace(*xlim, 200)
    ys = np.linspace(*ylim, 200)
    XX, YY = np.meshgrid(xs, ys)
    ZZ = np.clip(f(XX, YY), None, 50)
    axes[0].contourf(XX, YY, ZZ, levels=25, cmap='RdBu_r', alpha=0.5)
    axes[0].contour(XX, YY, ZZ, levels=25, colors='gray', linewidths=0.4, alpha=0.4)
    for name, (path, _) in results.items():
        axes[0].plot(path[:, 0], path[:, 1], '-', color=colors[name], lw=2, label=name)
        axes[0].plot(path[-1, 0], path[-1, 1], 'o', color=colors[name], ms=7)
    axes[0].scatter([x0], [y0], c='k', s=80, marker='*', zorder=5, label='Start')
    axes[0].set_xlim(xlim); axes[0].set_ylim(ylim)
    axes[0].set_xlabel('θ₁'); axes[0].set_ylabel('θ₂')
    axes[0].set_title('Trajectories', fontsize=11)
    axes[0].legend(fontsize=9)

    # Loss curves
    for name, (_, losses) in results.items():
        axes[1].semilogy(losses, color=colors[name], lw=2, label=f'{name} (final={losses[-1]:.2e})')
    axes[1].set_xlabel('Step'); axes[1].set_ylabel('Loss (log)')
    axes[1].set_title('Convergence', fontsize=11)
    axes[1].grid(True, which='both', alpha=0.3)
    axes[1].legend(fontsize=9)

    plt.suptitle(f'Surface: {surface}  κ={kappa}  steps={steps}', fontsize=10, y=0.0)
    plt.tight_layout()
    plt.show()


widgets.interact(
    draw_comparison,
    surface=widgets.Dropdown(options=['bowl', 'banana', 'saddle_valley'], value='bowl', description='Surface'),
    kappa=widgets.IntSlider(min=1, max=50, step=1, value=10, description='κ', continuous_update=False),
    steps=widgets.IntSlider(min=20, max=500, step=20, value=200, description='Steps', continuous_update=False),
    show_bgd=widgets.Checkbox(value=True, description='Batch GD'),
    lr_bgd=widgets.FloatSlider(min=0.001, max=0.3, step=0.005, value=0.03, description='α (BGD)', continuous_update=False),
    show_mom=widgets.Checkbox(value=True, description='Momentum'),
    lr_mom=widgets.FloatSlider(min=0.001, max=0.3, step=0.005, value=0.03, description='α (Mom)', continuous_update=False),
    beta=widgets.FloatSlider(min=0.5, max=0.99, step=0.01, value=0.9, description='β', continuous_update=False),
    show_ag=widgets.Checkbox(value=True, description='AdaGrad'),
    lr_ag=widgets.FloatSlider(min=0.01, max=2.0, step=0.05, value=0.5, description='α (AG)', continuous_update=False),
    show_rms=widgets.Checkbox(value=True, description='RMSProp'),
    lr_rms=widgets.FloatSlider(min=0.001, max=0.3, step=0.005, value=0.03, description='α (RMS)', continuous_update=False),
    rho=widgets.FloatSlider(min=0.5, max=0.999, step=0.01, value=0.9, description='ρ', continuous_update=False),
)